# All-sectors mangrove attribution using signed avoided EADs

This notebook extends the signed buildings workflow to all available sectors.

Method:
1. Keep all asset rows with non-zero avoided EAD.
2. For polygons and points, attribute directly from the original asset geometry.
3. For line assets, split geometries into shorter segments and allocate each asset's avoided EAD to segments by length share.
4. Attribute each analysis unit to mangroves within a 5000 m buffer using `area / distance` weights.
5. For units outside the 5000 m buffer, attribute to the nearest mangrove patch or nearest tied patches.
6. Summarize mangrove patches by positive, negative, net, and absolute attributed avoided EAD across all sectors and by sector.

Interpretation:
- Positive attributed values indicate avoided damages associated with mangroves.
- Negative attributed values indicate increased damages associated with mangroves in the underlying model outputs.

In [ ]:
from pathlib import Path
import pathlib
import sys

import geopandas
import matplotlib.pyplot as plt
import numpy
import pandas
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import FuncFormatter
from shapely.geometry import GeometryCollection, LineString, MultiLineString
from shapely.ops import substring

project_root = pathlib.Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
robyn_libraries_path = project_root / 'robyns_libraries'
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

In [ ]:
# User parameters
SCENARIO = 'minimum'  # 'minimum' or 'maximum'
BUFFER_M = 5000.0
KEEP_ONLY_NONZERO_AVOIDED = True
USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER = True
EDGE_SEGMENT_LENGTH_M = 1000.0
ZERO_DISTANCE_TOLERANCE_M = 0.001
MAP_DISPLAY_QUANTILE = 0.995

if SCENARIO not in {'minimum', 'maximum'}:
    raise ValueError("SCENARIO must be 'minimum' or 'maximum'.")
if BUFFER_M <= 0:
    raise ValueError('BUFFER_M must be > 0.')
if EDGE_SEGMENT_LENGTH_M <= 0:
    raise ValueError('EDGE_SEGMENT_LENGTH_M must be > 0.')
if ZERO_DISTANCE_TOLERANCE_M < 0:
    raise ValueError('ZERO_DISTANCE_TOLERANCE_M must be >= 0.')
if not (0 < MAP_DISPLAY_QUANTILE <= 1):
    raise ValueError('MAP_DISPLAY_QUANTILE must be within (0, 1].')

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / f'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_{SCENARIO}_scenario'
damage_estimates_path = results_path / 'damage_estimates'
asset_ead_csv = damage_estimates_path / 'coastal_ead_asset_level_usd.csv'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
mangrove_path = base_path / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
out_dir = damage_estimates_path / 'mangrove_attribution_area_distance_all_sectors_signed'
out_dir.mkdir(parents=True, exist_ok=True)

for required_path in [asset_ead_csv, network_csv, mangrove_path, jamaica_boundary_path]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required path: {required_path}')

method_label_base = f'signed_area_distance_{int(round(BUFFER_M))}m'
method_label = f'{method_label_base}_nn_fallback' if USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER else method_label_base

print(f'SCENARIO: {SCENARIO}')
print(f'BUFFER_M: {BUFFER_M:,.0f}')
print(f'KEEP_ONLY_NONZERO_AVOIDED: {KEEP_ONLY_NONZERO_AVOIDED}')
print(f'USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER: {USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER}')
print(f'EDGE_SEGMENT_LENGTH_M: {EDGE_SEGMENT_LENGTH_M:,.0f}')
print(f'Output folder: {out_dir}')

In [ ]:
# Load shared inputs
network_details = pandas.read_csv(network_csv)
required_network_columns = ['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column']
missing_network_columns = [column_name for column_name in required_network_columns if column_name not in network_details.columns]
if missing_network_columns:
    raise KeyError(f'Missing required network metadata columns: {missing_network_columns}')

network_details = network_details[required_network_columns].drop_duplicates().copy()
network_details = network_details.sort_values(['sector', 'asset_gpkg', 'asset_layer']).reset_index(drop=True)
network_details['geometry_file'] = network_details.apply(
    lambda row: damage_estimates_path / f"{row.asset_gpkg}_{row.asset_layer}_asset_damages_groupedby.gpkg",
    axis=1,
)

asset_ead = pandas.read_csv(asset_ead_csv)
if KEEP_ONLY_NONZERO_AVOIDED:
    asset_ead = asset_ead.loc[asset_ead['Avoided_EAD_USD'] != 0].copy()

asset_ead['Asset_ID'] = asset_ead['Asset_ID'].astype(str)
asset_ead['Avoided_EAD_Sign'] = numpy.where(
    asset_ead['Avoided_EAD_USD'] > 0,
    'positive',
    numpy.where(asset_ead['Avoided_EAD_USD'] < 0, 'negative', 'zero')
)

mangroves = geopandas.read_file(mangrove_path)
if mangroves.crs is None:
    raise ValueError('Mangrove CRS is missing.')
if str(mangroves.crs).upper() != 'EPSG:3448':
    mangroves = mangroves.to_crs('EPSG:3448')
if 'ID' in mangroves.columns:
    mangroves['Mangrove_ID'] = mangroves['ID'].astype(int)
else:
    mangroves = mangroves.reset_index(drop=True)
    mangroves['Mangrove_ID'] = numpy.arange(1, len(mangroves) + 1)
mangroves['Mangrove_Area_m2'] = mangroves.geometry.area
mangroves['Mangrove_Area_ha'] = mangroves['Mangrove_Area_m2'] / 10000.0
mangrove_base_columns = ['Mangrove_ID', 'Mangrove_Area_m2', 'Mangrove_Area_ha']
for mangrove_attribute_column in ['Parish', 'HECTARES', 'TYPE']:
    if mangrove_attribute_column in mangroves.columns:
        mangrove_base_columns.append(mangrove_attribute_column)

jamaica_boundary = geopandas.read_file(jamaica_boundary_path)
if jamaica_boundary.crs is None:
    raise ValueError('Jamaica boundary CRS is missing.')
if str(jamaica_boundary.crs).upper() != 'EPSG:3448':
    jamaica_boundary = jamaica_boundary.to_crs('EPSG:3448')

mangrove_lookup = pandas.DataFrame(mangroves[mangrove_base_columns].copy())
mangrove_lookup['mangrove_geometry'] = mangroves.geometry.values
mangrove_buffers = mangroves[mangrove_base_columns + ['geometry']].copy()
mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(BUFFER_M)

print('Network layers available:', len(network_details))
print('Asset EAD rows used:', len(asset_ead))
print('Mangrove patches:', len(mangroves))
display(network_details[['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column']])

In [ ]:
# Helper functions
asset_key_columns = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
unit_key_columns = asset_key_columns + ['Unit_ID']


def iterate_line_components(geometry):
    if geometry is None or geometry.is_empty:
        return []
    geometry_type = geometry.geom_type
    if geometry_type == 'LineString':
        return [geometry]
    if geometry_type == 'MultiLineString':
        return [line_component for line_component in geometry.geoms if not line_component.is_empty and line_component.length > 0]
    if geometry_type == 'GeometryCollection':
        line_components = []
        for geometry_part in geometry.geoms:
            line_components.extend(iterate_line_components(geometry_part))
        return line_components
    return []


def split_line_component(line_component, max_segment_length_m):
    if line_component.is_empty or line_component.length <= 0:
        return []
    segment_count = max(1, int(numpy.ceil(line_component.length / max_segment_length_m)))
    breakpoints = numpy.linspace(0.0, float(line_component.length), segment_count + 1)
    segments = []
    for segment_start, segment_end in zip(breakpoints[:-1], breakpoints[1:]):
        segment_geometry = substring(line_component, float(segment_start), float(segment_end))
        if not segment_geometry.is_empty and segment_geometry.length > 0:
            segments.append(segment_geometry)
    return segments


def split_line_geometry_to_segments(geometry, max_segment_length_m):
    segment_geometries = []
    for line_component in iterate_line_components(geometry):
        segment_geometries.extend(split_line_component(line_component, max_segment_length_m))
    return segment_geometries


def is_line_geometry(geometry):
    if geometry is None or geometry.is_empty:
        return False
    return geometry.geom_type in {'LineString', 'MultiLineString'}


def prepare_layer_asset_geometries(metadata_row):
    asset_subset = asset_ead.loc[
        (asset_ead['Asset'] == metadata_row.asset_gpkg) & (asset_ead['Layer'] == metadata_row.asset_layer),
        [
            'Sector',
            'Subsector',
            'Asset',
            'Layer',
            'Asset_ID',
            'EAD_With_Mangroves_USD',
            'EAD_Without_Mangroves_USD',
            'Avoided_EAD_USD',
            'Avoided_EAD_Share_of_Baseline',
            'Avoided_EAD_Sign',
        ],
    ].copy()
    if asset_subset.empty:
        return None

    geometry_file = metadata_row.geometry_file
    if not geometry_file.exists():
        return None

    geometry_gdf = geopandas.read_file(geometry_file)
    if geometry_gdf.crs is None:
        raise ValueError(f'CRS missing for geometry file: {geometry_file}')
    if str(geometry_gdf.crs).upper() != 'EPSG:3448':
        geometry_gdf = geometry_gdf.to_crs('EPSG:3448')
    if metadata_row.asset_id_column not in geometry_gdf.columns:
        raise KeyError(f"ID column '{metadata_row.asset_id_column}' missing in {geometry_file.name}")
    if len(geometry_gdf) == 0:
        return None

    geometry_gdf = geometry_gdf[[metadata_row.asset_id_column, 'geometry']].copy()
    geometry_gdf['Asset_ID'] = geometry_gdf[metadata_row.asset_id_column].astype(str)
    geometry_gdf = geometry_gdf.drop(columns=[metadata_row.asset_id_column])
    geometry_gdf = geometry_gdf.dropna(subset=['geometry']).copy()
    geometry_gdf = geopandas.GeoDataFrame(geometry_gdf, geometry='geometry', crs='EPSG:3448')

    merged_assets = geometry_gdf.merge(asset_subset, on='Asset_ID', how='inner')
    if merged_assets.empty:
        return None
    if merged_assets['Asset_ID'].duplicated().any():
        merged_assets = (
            geopandas.GeoDataFrame(merged_assets, geometry='geometry', crs='EPSG:3448')
            .dissolve(
                by=asset_key_columns,
                as_index=False,
                aggfunc={
                    'EAD_With_Mangroves_USD': 'first',
                    'EAD_Without_Mangroves_USD': 'first',
                    'Avoided_EAD_USD': 'first',
                    'Avoided_EAD_Share_of_Baseline': 'first',
                    'Avoided_EAD_Sign': 'first',
                },
            )
        )
    merged_assets['Original_Geometry_Type'] = merged_assets.geometry.geom_type
    return geopandas.GeoDataFrame(merged_assets, geometry='geometry', crs='EPSG:3448')


def build_analysis_units(layer_assets, metadata_row):
    analysis_unit_rows = []
    line_asset_count = 0
    segmented_unit_count = 0
    for asset_row in layer_assets.itertuples(index=False):
        asset_dict = asset_row._asdict()
        geometry = asset_dict.pop('geometry')
        original_geometry_type = asset_dict['Original_Geometry_Type']
        if geometry is None or geometry.is_empty:
            continue
        if is_line_geometry(geometry):
            line_asset_count += 1
            segment_geometries = split_line_geometry_to_segments(geometry, EDGE_SEGMENT_LENGTH_M)
            total_segment_length = float(sum(segment_geometry.length for segment_geometry in segment_geometries))
            if total_segment_length <= 0:
                continue
            for segment_index, segment_geometry in enumerate(segment_geometries, start=1):
                unit_share = float(segment_geometry.length / total_segment_length)
                analysis_unit_rows.append({
                    **asset_dict,
                    'Unit_ID': f"{asset_dict['Asset_ID']}__seg_{segment_index}",
                    'Unit_Geometry_Type': 'line_segment',
                    'Unit_Share_of_Asset': unit_share,
                    'Unit_Length_m': float(segment_geometry.length),
                    'Unit_Avoided_EAD_USD': float(asset_dict['Avoided_EAD_USD'] * unit_share),
                    'geometry': segment_geometry,
                })
            segmented_unit_count += len(segment_geometries)
        else:
            analysis_unit_rows.append({
                **asset_dict,
                'Unit_ID': str(asset_dict['Asset_ID']),
                'Unit_Geometry_Type': original_geometry_type,
                'Unit_Share_of_Asset': 1.0,
                'Unit_Length_m': numpy.nan,
                'Unit_Avoided_EAD_USD': float(asset_dict['Avoided_EAD_USD']),
                'geometry': geometry,
            })
    if not analysis_unit_rows:
        return None, line_asset_count, segmented_unit_count
    analysis_units = geopandas.GeoDataFrame(analysis_unit_rows, geometry='geometry', crs='EPSG:3448')
    return analysis_units, line_asset_count, segmented_unit_count


def apply_area_distance_weights(pair_gdf):
    if pair_gdf.empty:
        pair_gdf = pair_gdf.copy()
        pair_gdf['is_zero_distance'] = pandas.Series(dtype=bool)
        pair_gdf['has_zero_distance_match'] = pandas.Series(dtype=bool)
        pair_gdf['weight_raw'] = pandas.Series(dtype=float)
        pair_gdf['weight_sum'] = pandas.Series(dtype=float)
        pair_gdf['weight'] = pandas.Series(dtype=float)
        pair_gdf['Unit_Avoided_EAD_USD_attributed'] = pandas.Series(dtype=float)
        return pair_gdf

    pair_gdf = pair_gdf.copy()
    pair_gdf['is_zero_distance'] = pair_gdf['distance_m'] <= ZERO_DISTANCE_TOLERANCE_M
    pair_gdf['has_zero_distance_match'] = pair_gdf.groupby(unit_key_columns)['is_zero_distance'].transform('max').astype(bool)
    pair_gdf['weight_raw'] = 0.0

    zero_distance_rows = pair_gdf['has_zero_distance_match'] & pair_gdf['is_zero_distance']
    positive_distance_rows = (~pair_gdf['has_zero_distance_match']) & (pair_gdf['distance_m'] > ZERO_DISTANCE_TOLERANCE_M)

    pair_gdf.loc[zero_distance_rows, 'weight_raw'] = pair_gdf.loc[zero_distance_rows, 'Mangrove_Area_m2']
    pair_gdf.loc[positive_distance_rows, 'weight_raw'] = (
        pair_gdf.loc[positive_distance_rows, 'Mangrove_Area_m2']
        / pair_gdf.loc[positive_distance_rows, 'distance_m']
    )

    pair_gdf['weight_sum'] = pair_gdf.groupby(unit_key_columns)['weight_raw'].transform('sum')
    if (pair_gdf['weight_sum'] <= 0).any():
        bad_units = pair_gdf.loc[pair_gdf['weight_sum'] <= 0, unit_key_columns].drop_duplicates()
        raise ValueError(f'Found analysis units with non-positive weight sums: {len(bad_units):,}')

    pair_gdf['weight'] = pair_gdf['weight_raw'] / pair_gdf['weight_sum']
    pair_gdf['Unit_Avoided_EAD_USD_attributed'] = pair_gdf['Unit_Avoided_EAD_USD'] * pair_gdf['weight']
    pair_gdf['Attributed_EAD_Sign'] = numpy.where(
        pair_gdf['Unit_Avoided_EAD_USD_attributed'] > 0,
        'positive',
        numpy.where(pair_gdf['Unit_Avoided_EAD_USD_attributed'] < 0, 'negative', 'zero')
    )
    return pair_gdf

In [ ]:
# Run signed attribution across all sectors
layer_processing_rows = []
matched_pair_parts = []
asset_status_parts = []
analysis_unit_parts = []

for metadata_row in network_details.itertuples(index=False):
    print(f"\nProcessing: {metadata_row.sector} | {metadata_row.asset_description} | {metadata_row.asset_gpkg} | {metadata_row.asset_layer}")
    layer_assets = prepare_layer_asset_geometries(metadata_row)
    if layer_assets is None or layer_assets.empty:
        layer_processing_rows.append({
            'Sector': metadata_row.sector,
            'Subsector': metadata_row.asset_description,
            'Asset': metadata_row.asset_gpkg,
            'Layer': metadata_row.asset_layer,
            'Geometry_File': metadata_row.geometry_file.name,
            'Asset_Count': 0,
            'Analysis_Unit_Count': 0,
            'Line_Asset_Count': 0,
            'Segmented_Unit_Count': 0,
            'Buffer_Matched_Unit_Count': 0,
            'Fallback_Unit_Count': 0,
            'Attributed_Unit_Count': 0,
            'Input_Net_Avoided_EAD_USD': 0.0,
            'Input_Positive_Avoided_EAD_USD': 0.0,
            'Input_Negative_Avoided_EAD_USD': 0.0,
            'Attributed_Net_Avoided_EAD_USD': 0.0,
            'Attributed_Positive_Avoided_EAD_USD': 0.0,
            'Attributed_Negative_Avoided_EAD_USD': 0.0,
        })
        print(' - skipped (no matching asset geometries or no non-zero avoided EAD rows)')
        continue

    analysis_units, line_asset_count, segmented_unit_count = build_analysis_units(layer_assets, metadata_row)
    if analysis_units is None or analysis_units.empty:
        print(' - skipped (no analysis units produced)')
        continue
    analysis_units = geopandas.GeoDataFrame(analysis_units, geometry='geometry', crs='EPSG:3448')
    analysis_unit_parts.append(analysis_units.copy())

    candidate_pairs = geopandas.sjoin(
        analysis_units,
        mangrove_buffers[['Mangrove_ID', 'geometry']],
        how='left',
        predicate='intersects',
    )
    candidate_pairs['nearby_mangrove_count'] = candidate_pairs.groupby(unit_key_columns)['Mangrove_ID'].transform(
        lambda matched_mangrove_ids: matched_mangrove_ids.notna().sum()
    )
    candidate_pairs['nearby_mangrove_count'] = candidate_pairs['nearby_mangrove_count'].fillna(0).astype(int)

    buffer_pairs = candidate_pairs.dropna(subset=['Mangrove_ID']).copy()
    if not buffer_pairs.empty:
        buffer_pairs['Mangrove_ID'] = buffer_pairs['Mangrove_ID'].astype(int)
        buffer_pairs = buffer_pairs.merge(mangrove_lookup, on='Mangrove_ID', how='left')
        mangrove_geometry_series = geopandas.GeoSeries(buffer_pairs['mangrove_geometry'], index=buffer_pairs.index, crs='EPSG:3448')
        buffer_pairs['distance_m'] = buffer_pairs.geometry.distance(mangrove_geometry_series)
        buffer_pairs = apply_area_distance_weights(buffer_pairs)
        buffer_pairs['Attribution_Source'] = 'buffer_area_distance'
        buffer_pairs['nearest_tie_count'] = 0
    else:
        buffer_pairs = geopandas.GeoDataFrame(buffer_pairs, geometry='geometry', crs='EPSG:3448')
        buffer_pairs['distance_m'] = pandas.Series(dtype=float)
        buffer_pairs = apply_area_distance_weights(buffer_pairs)
        buffer_pairs['Attribution_Source'] = pandas.Series(dtype=str)
        buffer_pairs['nearest_tie_count'] = pandas.Series(dtype=int)

    buffer_matched_unit_ids = set(buffer_pairs['Unit_ID'].astype(str).unique())
    unmatched_units = analysis_units.loc[~analysis_units['Unit_ID'].isin(buffer_matched_unit_ids)].copy()

    fallback_pairs = pandas.DataFrame(columns=[])
    if USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER and not unmatched_units.empty:
        fallback_pairs = geopandas.sjoin_nearest(
            unmatched_units,
            mangroves[mangrove_base_columns + ['geometry']],
            how='left',
            distance_col='distance_m',
        )
        fallback_pairs = fallback_pairs.dropna(subset=['Mangrove_ID']).copy()
        fallback_pairs['Mangrove_ID'] = fallback_pairs['Mangrove_ID'].astype(int)
        fallback_pairs['nearby_mangrove_count'] = 0
        fallback_pairs['nearest_tie_count'] = fallback_pairs.groupby(unit_key_columns)['Mangrove_ID'].transform('size').astype(int)
        fallback_pairs = apply_area_distance_weights(fallback_pairs)
        fallback_pairs['Attribution_Source'] = 'nearest_outside_buffer'

    combined_pair_columns = [
        'Sector',
        'Subsector',
        'Asset',
        'Layer',
        'Asset_ID',
        'EAD_With_Mangroves_USD',
        'EAD_Without_Mangroves_USD',
        'Avoided_EAD_USD',
        'Avoided_EAD_Share_of_Baseline',
        'Avoided_EAD_Sign',
        'Original_Geometry_Type',
        'Unit_ID',
        'Unit_Geometry_Type',
        'Unit_Share_of_Asset',
        'Unit_Length_m',
        'Unit_Avoided_EAD_USD',
        'geometry',
        'Mangrove_ID',
        'Mangrove_Area_m2',
        'Mangrove_Area_ha',
        'distance_m',
        'nearby_mangrove_count',
        'nearest_tie_count',
        'is_zero_distance',
        'has_zero_distance_match',
        'weight_raw',
        'weight_sum',
        'weight',
        'Unit_Avoided_EAD_USD_attributed',
        'Attributed_EAD_Sign',
        'Attribution_Source',
    ]
    empty_pair_table = pandas.DataFrame(columns=combined_pair_columns)
    matched_pairs = pandas.concat(
        [
            buffer_pairs[combined_pair_columns] if len(buffer_pairs) > 0 else empty_pair_table,
            fallback_pairs[combined_pair_columns] if len(fallback_pairs) > 0 else empty_pair_table,
        ],
        ignore_index=True,
    )
    matched_pairs = geopandas.GeoDataFrame(matched_pairs, geometry='geometry', crs='EPSG:3448')
    matched_pair_parts.append(matched_pairs.copy())

    unit_weight_check = (
        matched_pairs.groupby(unit_key_columns, as_index=False)['weight']
        .sum()
        .rename(columns={'weight': 'weight_sum_check'})
    ) if len(matched_pairs) > 0 else pandas.DataFrame(columns=unit_key_columns + ['weight_sum_check'])
    unit_attribution = (
        matched_pairs.groupby(unit_key_columns, as_index=False)['Unit_Avoided_EAD_USD_attributed']
        .sum()
        .rename(columns={'Unit_Avoided_EAD_USD_attributed': 'Attributed_Unit_Avoided_EAD_USD'})
    ) if len(matched_pairs) > 0 else pandas.DataFrame(columns=unit_key_columns + ['Attributed_Unit_Avoided_EAD_USD'])

    buffer_unit_status = buffer_pairs[unit_key_columns].drop_duplicates().copy()
    buffer_unit_status['Has_Buffer_Mangrove'] = 1
    fallback_unit_status = fallback_pairs[unit_key_columns].drop_duplicates().copy() if len(fallback_pairs) > 0 else pandas.DataFrame(columns=unit_key_columns)
    fallback_unit_status['Used_Nearest_Fallback'] = 1

    unit_status = analysis_units[asset_key_columns + ['Unit_ID', 'Unit_Avoided_EAD_USD', 'geometry']].copy()
    unit_status = unit_status.merge(unit_weight_check, on=unit_key_columns, how='left')
    unit_status = unit_status.merge(unit_attribution, on=unit_key_columns, how='left')
    unit_status = unit_status.merge(buffer_unit_status, on=unit_key_columns, how='left')
    unit_status = unit_status.merge(fallback_unit_status, on=unit_key_columns, how='left')
    unit_status['weight_sum_check'] = unit_status['weight_sum_check'].fillna(0.0)
    unit_status['Attributed_Unit_Avoided_EAD_USD'] = unit_status['Attributed_Unit_Avoided_EAD_USD'].fillna(0.0)
    unit_status['Has_Buffer_Mangrove'] = unit_status['Has_Buffer_Mangrove'].fillna(0).astype(int)
    unit_status['Used_Nearest_Fallback'] = unit_status['Used_Nearest_Fallback'].fillna(0).astype(int)

    asset_status = (
        unit_status.groupby(asset_key_columns, as_index=False)
        .agg(
            Avoided_EAD_USD=('Unit_Avoided_EAD_USD', 'sum'),
            Attributed_EAD_USD=('Attributed_Unit_Avoided_EAD_USD', 'sum'),
            Has_Buffer_Mangrove=('Has_Buffer_Mangrove', 'max'),
            Used_Nearest_Fallback=('Used_Nearest_Fallback', 'max'),
            Analysis_Unit_Count=('Unit_ID', 'nunique'),
        )
    )
    asset_status['Has_Attributed_Mangrove'] = (asset_status['Attributed_EAD_USD'] != 0).astype(int)
    asset_status['Unattributed_EAD_USD'] = asset_status['Avoided_EAD_USD'] - asset_status['Attributed_EAD_USD']
    asset_status['Avoided_EAD_Sign'] = numpy.where(
        asset_status['Avoided_EAD_USD'] > 0,
        'positive',
        numpy.where(asset_status['Avoided_EAD_USD'] < 0, 'negative', 'zero')
    )
    asset_status['Attributed_EAD_Sign'] = numpy.where(
        asset_status['Attributed_EAD_USD'] > 0,
        'positive',
        numpy.where(asset_status['Attributed_EAD_USD'] < 0, 'negative', 'zero')
    )
    asset_status_parts.append(asset_status.copy())

    layer_processing_rows.append({
        'Sector': metadata_row.sector,
        'Subsector': metadata_row.asset_description,
        'Asset': metadata_row.asset_gpkg,
        'Layer': metadata_row.asset_layer,
        'Geometry_File': metadata_row.geometry_file.name,
        'Asset_Count': int(len(layer_assets)),
        'Analysis_Unit_Count': int(len(analysis_units)),
        'Line_Asset_Count': int(line_asset_count),
        'Segmented_Unit_Count': int(segmented_unit_count),
        'Buffer_Matched_Unit_Count': int(buffer_pairs['Unit_ID'].nunique()) if len(buffer_pairs) > 0 else 0,
        'Fallback_Unit_Count': int(fallback_pairs['Unit_ID'].nunique()) if len(fallback_pairs) > 0 else 0,
        'Attributed_Unit_Count': int(matched_pairs['Unit_ID'].nunique()) if len(matched_pairs) > 0 else 0,
        'Input_Net_Avoided_EAD_USD': float(asset_status['Avoided_EAD_USD'].sum()),
        'Input_Positive_Avoided_EAD_USD': float(asset_status.loc[asset_status['Avoided_EAD_USD'] > 0, 'Avoided_EAD_USD'].sum()),
        'Input_Negative_Avoided_EAD_USD': float(asset_status.loc[asset_status['Avoided_EAD_USD'] < 0, 'Avoided_EAD_USD'].sum()),
        'Attributed_Net_Avoided_EAD_USD': float(asset_status['Attributed_EAD_USD'].sum()),
        'Attributed_Positive_Avoided_EAD_USD': float(asset_status.loc[asset_status['Attributed_EAD_USD'] > 0, 'Attributed_EAD_USD'].sum()),
        'Attributed_Negative_Avoided_EAD_USD': float(asset_status.loc[asset_status['Attributed_EAD_USD'] < 0, 'Attributed_EAD_USD'].sum()),
    })
    print(f" - assets: {len(layer_assets):,} | analysis units: {len(analysis_units):,} | buffer units: {layer_processing_rows[-1]['Buffer_Matched_Unit_Count']:,} | fallback units: {layer_processing_rows[-1]['Fallback_Unit_Count']:,}")

if not matched_pair_parts:
    raise ValueError('No matched pairs were created across any sector.')

matched_pairs_all = geopandas.GeoDataFrame(pandas.concat(matched_pair_parts, ignore_index=True), geometry='geometry', crs='EPSG:3448')
asset_status_all = pandas.concat(asset_status_parts, ignore_index=True) if asset_status_parts else pandas.DataFrame()
analysis_units_all = geopandas.GeoDataFrame(pandas.concat(analysis_unit_parts, ignore_index=True), geometry='geometry', crs='EPSG:3448') if analysis_unit_parts else geopandas.GeoDataFrame()
layer_processing_summary = pandas.DataFrame(layer_processing_rows)

print('\nFinished all-sector attribution.')
print('Matched pair rows:', len(matched_pairs_all))
print('Asset status rows:', len(asset_status_all))
print('Analysis units:', len(analysis_units_all))
display(layer_processing_summary)

In [ ]:
# QA checks and summary totals across all sectors
max_weight_error = 0.0
if len(matched_pairs_all) > 0:
    unit_weight_totals = matched_pairs_all.groupby(unit_key_columns)['weight'].sum()
    max_weight_error = float((unit_weight_totals - 1.0).abs().max())

input_net_total_usd = float(asset_status_all['Avoided_EAD_USD'].sum())
input_positive_total_usd = float(asset_status_all.loc[asset_status_all['Avoided_EAD_USD'] > 0, 'Avoided_EAD_USD'].sum())
input_negative_total_usd = float(asset_status_all.loc[asset_status_all['Avoided_EAD_USD'] < 0, 'Avoided_EAD_USD'].sum())
attributed_net_total_usd = float(asset_status_all['Attributed_EAD_USD'].sum())
attributed_positive_total_usd = float(asset_status_all.loc[asset_status_all['Attributed_EAD_USD'] > 0, 'Attributed_EAD_USD'].sum())
attributed_negative_total_usd = float(asset_status_all.loc[asset_status_all['Attributed_EAD_USD'] < 0, 'Attributed_EAD_USD'].sum())
unattributed_net_total_usd = float(asset_status_all['Unattributed_EAD_USD'].sum())

print(f'Input net avoided EAD (USD): {input_net_total_usd:,.2f}')
print(f'Input positive avoided EAD (USD): {input_positive_total_usd:,.2f}')
print(f'Input negative avoided EAD (USD): {input_negative_total_usd:,.2f}')
print(f'Attributed net avoided EAD (USD): {attributed_net_total_usd:,.2f}')
print(f'Attributed positive avoided EAD (USD): {attributed_positive_total_usd:,.2f}')
print(f'Attributed negative avoided EAD (USD): {attributed_negative_total_usd:,.2f}')
print(f'Unattributed net avoided EAD (USD): {unattributed_net_total_usd:,.12f}')
print(f'Max unit weight-sum error: {max_weight_error:.12f}')

if not numpy.isclose(input_net_total_usd, attributed_net_total_usd + unattributed_net_total_usd, atol=1e-6):
    raise ValueError('Input net total does not equal attributed plus unattributed totals.')

sector_summary = (
    asset_status_all.groupby('Sector', as_index=False)
    .agg(
        Asset_Count=('Asset_ID', 'size'),
        Buffer_Asset_Count=('Has_Buffer_Mangrove', 'sum'),
        Fallback_Asset_Count=('Used_Nearest_Fallback', 'sum'),
        Net_Avoided_EAD_USD=('Avoided_EAD_USD', 'sum'),
        Positive_Avoided_EAD_USD=('Avoided_EAD_USD', lambda values: float(values[values > 0].sum())),
        Negative_Avoided_EAD_USD=('Avoided_EAD_USD', lambda values: float(values[values < 0].sum())),
        Attributed_Net_Avoided_EAD_USD=('Attributed_EAD_USD', 'sum'),
    )
    .sort_values('Attributed_Net_Avoided_EAD_USD', ascending=False)
    .reset_index(drop=True)
)
display(sector_summary)

In [ ]:
# Build output tables and save files
unit_to_mangrove_output = matched_pairs_all[[
    'Sector',
    'Subsector',
    'Asset',
    'Layer',
    'Asset_ID',
    'Avoided_EAD_Sign',
    'Original_Geometry_Type',
    'Unit_ID',
    'Unit_Geometry_Type',
    'Unit_Share_of_Asset',
    'Unit_Length_m',
    'Avoided_EAD_USD',
    'Unit_Avoided_EAD_USD',
    'Mangrove_ID',
    'Mangrove_Area_m2',
    'Mangrove_Area_ha',
    'distance_m',
    'nearby_mangrove_count',
    'nearest_tie_count',
    'weight_raw',
    'weight',
    'Unit_Avoided_EAD_USD_attributed',
    'Attributed_EAD_Sign',
    'Attribution_Source',
]].copy()
unit_to_mangrove_output['Positive_Avoided_EAD_USD_attributed'] = unit_to_mangrove_output['Unit_Avoided_EAD_USD_attributed'].clip(lower=0.0)
unit_to_mangrove_output['Negative_Avoided_EAD_USD_attributed'] = unit_to_mangrove_output['Unit_Avoided_EAD_USD_attributed'].clip(upper=0.0)
unit_to_mangrove_output['Absolute_Avoided_EAD_USD_attributed'] = unit_to_mangrove_output['Unit_Avoided_EAD_USD_attributed'].abs()

asset_to_mangrove_output = (
    unit_to_mangrove_output.groupby(['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Mangrove_ID'], as_index=False)
    .agg(
        Asset_Avoided_EAD_USD=('Avoided_EAD_USD', 'first'),
        Analysis_Unit_Count=('Unit_ID', 'nunique'),
        Mangrove_Area_m2=('Mangrove_Area_m2', 'first'),
        Mangrove_Area_ha=('Mangrove_Area_ha', 'first'),
        Min_Distance_m=('distance_m', 'min'),
        Mean_Distance_m=('distance_m', 'mean'),
        Max_Distance_m=('distance_m', 'max'),
        Buffer_Analysis_Unit_Count=('Attribution_Source', lambda values: int((pandas.Series(values) == 'buffer_area_distance').sum())),
        Fallback_Analysis_Unit_Count=('Attribution_Source', lambda values: int((pandas.Series(values) == 'nearest_outside_buffer').sum())),
        Avoided_EAD_USD_attributed=('Unit_Avoided_EAD_USD_attributed', 'sum'),
        Positive_Avoided_EAD_USD_attributed=('Positive_Avoided_EAD_USD_attributed', 'sum'),
        Negative_Avoided_EAD_USD_attributed=('Negative_Avoided_EAD_USD_attributed', 'sum'),
        Absolute_Avoided_EAD_USD_attributed=('Absolute_Avoided_EAD_USD_attributed', 'sum'),
    )
)
asset_to_mangrove_output['Attributed_EAD_Sign'] = numpy.where(
    asset_to_mangrove_output['Avoided_EAD_USD_attributed'] > 0,
    'positive',
    numpy.where(asset_to_mangrove_output['Avoided_EAD_USD_attributed'] < 0, 'negative', 'zero')
)

sector_attribution_breakdown = (
    asset_status_all.groupby('Sector', as_index=False)
    .agg(
        Avoided_EAD_USD=('Avoided_EAD_USD', 'sum'),
        Attributed_EAD_USD=('Attributed_EAD_USD', 'sum'),
        Unattributed_EAD_USD=('Unattributed_EAD_USD', 'sum'),
        Asset_Count=('Asset_ID', 'size'),
    )
)
sector_attribution_breakdown['Attributed_share_percent'] = numpy.where(
    sector_attribution_breakdown['Avoided_EAD_USD'] != 0,
    100.0 * sector_attribution_breakdown['Attributed_EAD_USD'] / sector_attribution_breakdown['Avoided_EAD_USD'],
    numpy.nan,
)
sector_attribution_breakdown['Unattributed_share_percent'] = numpy.where(
    sector_attribution_breakdown['Avoided_EAD_USD'] != 0,
    100.0 * sector_attribution_breakdown['Unattributed_EAD_USD'] / sector_attribution_breakdown['Avoided_EAD_USD'],
    numpy.nan,
)
sector_attribution_breakdown = sector_attribution_breakdown.sort_values('Sector').reset_index(drop=True)

subsector_attribution_breakdown = (
    asset_status_all.groupby(['Sector', 'Subsector'], as_index=False)
    .agg(
        Avoided_EAD_USD=('Avoided_EAD_USD', 'sum'),
        Attributed_EAD_USD=('Attributed_EAD_USD', 'sum'),
        Unattributed_EAD_USD=('Unattributed_EAD_USD', 'sum'),
        Asset_Count=('Asset_ID', 'size'),
    )
)
subsector_attribution_breakdown['Attributed_share_percent'] = numpy.where(
    subsector_attribution_breakdown['Avoided_EAD_USD'] != 0,
    100.0 * subsector_attribution_breakdown['Attributed_EAD_USD'] / subsector_attribution_breakdown['Avoided_EAD_USD'],
    numpy.nan,
)
subsector_attribution_breakdown['Unattributed_share_percent'] = numpy.where(
    subsector_attribution_breakdown['Avoided_EAD_USD'] != 0,
    100.0 * subsector_attribution_breakdown['Unattributed_EAD_USD'] / subsector_attribution_breakdown['Avoided_EAD_USD'],
    numpy.nan,
)
subsector_attribution_breakdown = subsector_attribution_breakdown.sort_values(['Sector', 'Subsector']).reset_index(drop=True)


asset_signed_profile_input = asset_status_all.assign(
    Positive_Avoided_EAD_USD=asset_status_all['Avoided_EAD_USD'].clip(lower=0.0),
    Negative_Avoided_EAD_USD=asset_status_all['Avoided_EAD_USD'].clip(upper=0.0),
    Positive_Attributed_EAD_USD=asset_status_all['Attributed_EAD_USD'].clip(lower=0.0),
    Negative_Attributed_EAD_USD=asset_status_all['Attributed_EAD_USD'].clip(upper=0.0),
    Absolute_Attributed_EAD_USD=asset_status_all['Attributed_EAD_USD'].abs(),
)

sector_signed_profile = (
    asset_signed_profile_input.groupby('Sector', as_index=False)
    .agg(
        Asset_Count=('Asset_ID', 'size'),
        Positive_Asset_Count=('Attributed_EAD_USD', lambda values: int((values > 0).sum())),
        Negative_Asset_Count=('Attributed_EAD_USD', lambda values: int((values < 0).sum())),
        Zero_Asset_Count=('Attributed_EAD_USD', lambda values: int((values == 0).sum())),
        Positive_Avoided_EAD_USD=('Positive_Avoided_EAD_USD', 'sum'),
        Negative_Avoided_EAD_USD=('Negative_Avoided_EAD_USD', 'sum'),
        Net_Avoided_EAD_USD=('Avoided_EAD_USD', 'sum'),
        Positive_Attributed_EAD_USD=('Positive_Attributed_EAD_USD', 'sum'),
        Negative_Attributed_EAD_USD=('Negative_Attributed_EAD_USD', 'sum'),
        Net_Attributed_EAD_USD=('Attributed_EAD_USD', 'sum'),
        Absolute_Attributed_EAD_USD=('Absolute_Attributed_EAD_USD', 'sum'),
    )
)

subsector_signed_profile = (
    asset_signed_profile_input.groupby(['Sector', 'Subsector'], as_index=False)
    .agg(
        Asset_Count=('Asset_ID', 'size'),
        Positive_Asset_Count=('Attributed_EAD_USD', lambda values: int((values > 0).sum())),
        Negative_Asset_Count=('Attributed_EAD_USD', lambda values: int((values < 0).sum())),
        Zero_Asset_Count=('Attributed_EAD_USD', lambda values: int((values == 0).sum())),
        Positive_Avoided_EAD_USD=('Positive_Avoided_EAD_USD', 'sum'),
        Negative_Avoided_EAD_USD=('Negative_Avoided_EAD_USD', 'sum'),
        Net_Avoided_EAD_USD=('Avoided_EAD_USD', 'sum'),
        Positive_Attributed_EAD_USD=('Positive_Attributed_EAD_USD', 'sum'),
        Negative_Attributed_EAD_USD=('Negative_Attributed_EAD_USD', 'sum'),
        Net_Attributed_EAD_USD=('Attributed_EAD_USD', 'sum'),
        Absolute_Attributed_EAD_USD=('Absolute_Attributed_EAD_USD', 'sum'),
    )
)

for signed_profile_table in [sector_signed_profile, subsector_signed_profile]:
    signed_profile_table['Negative_Avoided_EAD_USD_abs'] = -signed_profile_table['Negative_Avoided_EAD_USD']
    signed_profile_table['Negative_Attributed_EAD_USD_abs'] = -signed_profile_table['Negative_Attributed_EAD_USD']
    signed_profile_table['Gross_Avoided_EAD_USD'] = (
        signed_profile_table['Positive_Avoided_EAD_USD'] + signed_profile_table['Negative_Avoided_EAD_USD_abs']
    )
    signed_profile_table['Gross_Attributed_EAD_USD'] = (
        signed_profile_table['Positive_Attributed_EAD_USD'] + signed_profile_table['Negative_Attributed_EAD_USD_abs']
    )
    signed_profile_table['Positive_share_of_gross_avoided_pct'] = numpy.where(
        signed_profile_table['Gross_Avoided_EAD_USD'] > 0,
        100.0 * signed_profile_table['Positive_Avoided_EAD_USD'] / signed_profile_table['Gross_Avoided_EAD_USD'],
        numpy.nan,
    )
    signed_profile_table['Negative_share_of_gross_avoided_pct'] = numpy.where(
        signed_profile_table['Gross_Avoided_EAD_USD'] > 0,
        100.0 * signed_profile_table['Negative_Avoided_EAD_USD_abs'] / signed_profile_table['Gross_Avoided_EAD_USD'],
        numpy.nan,
    )
    signed_profile_table['Positive_share_of_gross_attributed_pct'] = numpy.where(
        signed_profile_table['Gross_Attributed_EAD_USD'] > 0,
        100.0 * signed_profile_table['Positive_Attributed_EAD_USD'] / signed_profile_table['Gross_Attributed_EAD_USD'],
        numpy.nan,
    )
    signed_profile_table['Negative_share_of_gross_attributed_pct'] = numpy.where(
        signed_profile_table['Gross_Attributed_EAD_USD'] > 0,
        100.0 * signed_profile_table['Negative_Attributed_EAD_USD_abs'] / signed_profile_table['Gross_Attributed_EAD_USD'],
        numpy.nan,
    )
    signed_profile_table['Net_retention_of_positive_attributed_pct'] = numpy.where(
        signed_profile_table['Positive_Attributed_EAD_USD'] > 0,
        100.0 * signed_profile_table['Net_Attributed_EAD_USD'] / signed_profile_table['Positive_Attributed_EAD_USD'],
        numpy.nan,
    )
    signed_profile_table['Negative_drag_on_positive_attributed_pct'] = numpy.where(
        signed_profile_table['Positive_Attributed_EAD_USD'] > 0,
        100.0 * signed_profile_table['Negative_Attributed_EAD_USD_abs'] / signed_profile_table['Positive_Attributed_EAD_USD'],
        numpy.nan,
    )
    signed_profile_table['Has_Mixed_Positive_And_Negative_Assets'] = (
        (signed_profile_table['Positive_Asset_Count'] > 0)
        & (signed_profile_table['Negative_Asset_Count'] > 0)
    )
    signed_profile_table['Net_Impact_Class'] = numpy.select(
        [
            signed_profile_table['Net_Attributed_EAD_USD'] > 0,
            signed_profile_table['Net_Attributed_EAD_USD'] < 0,
        ],
        ['net_protective', 'net_damage_increasing'],
        default='net_zero',
    )

sector_signed_profile = sector_signed_profile.sort_values('Net_Attributed_EAD_USD', ascending=False).reset_index(drop=True)
subsector_signed_profile = subsector_signed_profile.sort_values(
    ['Net_Attributed_EAD_USD', 'Sector', 'Subsector'],
    ascending=[False, True, True],
).reset_index(drop=True)

mangrove_sector_summary = (
    unit_to_mangrove_output.groupby(['Mangrove_ID', 'Sector'], as_index=False)
    .agg(
        Net_Avoided_EAD_USD_attributed=('Unit_Avoided_EAD_USD_attributed', 'sum'),
        Positive_Avoided_EAD_USD_attributed=('Positive_Avoided_EAD_USD_attributed', 'sum'),
        Negative_Avoided_EAD_USD_attributed=('Negative_Avoided_EAD_USD_attributed', 'sum'),
        Absolute_Avoided_EAD_USD_attributed=('Absolute_Avoided_EAD_USD_attributed', 'sum'),
        Asset_Count=('Asset_ID', 'nunique'),
        Analysis_Unit_Count=('Unit_ID', 'nunique'),
    )
)

mangrove_total_summary = (
    unit_to_mangrove_output.groupby('Mangrove_ID', as_index=False)
    .agg(
        Net_Avoided_EAD_USD_attributed=('Unit_Avoided_EAD_USD_attributed', 'sum'),
        Positive_Avoided_EAD_USD_attributed=('Positive_Avoided_EAD_USD_attributed', 'sum'),
        Negative_Avoided_EAD_USD_attributed=('Negative_Avoided_EAD_USD_attributed', 'sum'),
        Absolute_Avoided_EAD_USD_attributed=('Absolute_Avoided_EAD_USD_attributed', 'sum'),
        Asset_Count=('Asset_ID', 'nunique'),
        Analysis_Unit_Count=('Unit_ID', 'nunique'),
        Mean_Distance_m=('distance_m', 'mean'),
    )
    .sort_values('Net_Avoided_EAD_USD_attributed', ascending=False)
    .reset_index(drop=True)
)
mangrove_total_summary['Negative_Avoided_EAD_USD_attributed_abs'] = -mangrove_total_summary['Negative_Avoided_EAD_USD_attributed']

count_specs = [
    ('Positive_Asset_Count', unit_to_mangrove_output['Unit_Avoided_EAD_USD_attributed'] > 0),
    ('Negative_Asset_Count', unit_to_mangrove_output['Unit_Avoided_EAD_USD_attributed'] < 0),
    ('Buffer_Analysis_Unit_Count', unit_to_mangrove_output['Attribution_Source'] == 'buffer_area_distance'),
    ('Fallback_Analysis_Unit_Count', unit_to_mangrove_output['Attribution_Source'] == 'nearest_outside_buffer'),
]
for count_column_name, row_filter in count_specs:
    count_table = (
        unit_to_mangrove_output.loc[row_filter, ['Mangrove_ID', 'Unit_ID']]
        .drop_duplicates()
        .groupby('Mangrove_ID')
        .size()
        .rename(count_column_name)
        .reset_index()
    )
    mangrove_total_summary = mangrove_total_summary.merge(count_table, on='Mangrove_ID', how='left')
for count_column_name, _ in count_specs:
    mangrove_total_summary[count_column_name] = mangrove_total_summary[count_column_name].fillna(0).astype(int)


mangrove_asset_sign_counts = (
    asset_to_mangrove_output.groupby('Mangrove_ID', as_index=False)
    .agg(
        Positive_Attributed_Asset_Count=('Avoided_EAD_USD_attributed', lambda values: int((values > 0).sum())),
        Negative_Attributed_Asset_Count=('Avoided_EAD_USD_attributed', lambda values: int((values < 0).sum())),
        Zero_Attributed_Asset_Count=('Avoided_EAD_USD_attributed', lambda values: int((values == 0).sum())),
    )
)

mangrove_signed_profile = mangrove_total_summary.merge(mangrove_asset_sign_counts, on='Mangrove_ID', how='left')
for count_column_name in ['Positive_Attributed_Asset_Count', 'Negative_Attributed_Asset_Count', 'Zero_Attributed_Asset_Count']:
    mangrove_signed_profile[count_column_name] = mangrove_signed_profile[count_column_name].fillna(0).astype(int)
mangrove_signed_profile['Gross_Avoided_EAD_USD_attributed'] = (
    mangrove_signed_profile['Positive_Avoided_EAD_USD_attributed']
    + mangrove_signed_profile['Negative_Avoided_EAD_USD_attributed_abs']
)
mangrove_signed_profile['Positive_share_of_gross_attributed_pct'] = numpy.where(
    mangrove_signed_profile['Gross_Avoided_EAD_USD_attributed'] > 0,
    100.0 * mangrove_signed_profile['Positive_Avoided_EAD_USD_attributed'] / mangrove_signed_profile['Gross_Avoided_EAD_USD_attributed'],
    numpy.nan,
)
mangrove_signed_profile['Negative_share_of_gross_attributed_pct'] = numpy.where(
    mangrove_signed_profile['Gross_Avoided_EAD_USD_attributed'] > 0,
    100.0 * mangrove_signed_profile['Negative_Avoided_EAD_USD_attributed_abs'] / mangrove_signed_profile['Gross_Avoided_EAD_USD_attributed'],
    numpy.nan,
)
mangrove_signed_profile['Net_retention_of_positive_attributed_pct'] = numpy.where(
    mangrove_signed_profile['Positive_Avoided_EAD_USD_attributed'] > 0,
    100.0 * mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] / mangrove_signed_profile['Positive_Avoided_EAD_USD_attributed'],
    numpy.nan,
)
mangrove_signed_profile['Negative_drag_on_positive_attributed_pct'] = numpy.where(
    mangrove_signed_profile['Positive_Avoided_EAD_USD_attributed'] > 0,
    100.0 * mangrove_signed_profile['Negative_Avoided_EAD_USD_attributed_abs'] / mangrove_signed_profile['Positive_Avoided_EAD_USD_attributed'],
    numpy.nan,
)
mangrove_signed_profile['Has_Mixed_Positive_And_Negative_Assets'] = (
    (mangrove_signed_profile['Positive_Attributed_Asset_Count'] > 0)
    & (mangrove_signed_profile['Negative_Attributed_Asset_Count'] > 0)
)
mangrove_signed_profile['Net_Impact_Class'] = numpy.select(
    [
        mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] > 0,
        mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] < 0,
    ],
    ['net_protective', 'net_damage_increasing'],
    default='net_zero',
)
mangrove_signed_profile = mangrove_signed_profile.sort_values(
    ['Net_Avoided_EAD_USD_attributed', 'Mangrove_ID'],
    ascending=[False, True],
).reset_index(drop=True)

total_mangrove_patch_count = int(len(mangrove_signed_profile))
net_protective_patch_count = int((mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] > 0).sum())
net_negative_patch_count = int((mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] < 0).sum())
net_zero_patch_count = int((mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] == 0).sum())
mixed_sign_patch_count = int(mangrove_signed_profile['Has_Mixed_Positive_And_Negative_Assets'].sum())
mixed_sign_net_protective_patch_count = int(((mangrove_signed_profile['Has_Mixed_Positive_And_Negative_Assets']) & (mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] > 0)).sum())
mixed_sign_net_negative_patch_count = int(((mangrove_signed_profile['Has_Mixed_Positive_And_Negative_Assets']) & (mangrove_signed_profile['Net_Avoided_EAD_USD_attributed'] < 0)).sum())
positive_only_patch_count = int(((mangrove_signed_profile['Positive_Attributed_Asset_Count'] > 0) & (mangrove_signed_profile['Negative_Attributed_Asset_Count'] == 0)).sum())
negative_only_patch_count = int(((mangrove_signed_profile['Positive_Attributed_Asset_Count'] == 0) & (mangrove_signed_profile['Negative_Attributed_Asset_Count'] > 0)).sum())

mangrove_patch_net_sign_summary = pandas.DataFrame([
    {
        'Scenario': SCENARIO,
        'Total_Mangrove_Patch_Count': total_mangrove_patch_count,
        'Net_Protective_Patch_Count': net_protective_patch_count,
        'Net_Protective_Patch_pct': (100.0 * net_protective_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Net_Negative_Patch_Count': net_negative_patch_count,
        'Net_Negative_Patch_pct': (100.0 * net_negative_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Net_Zero_Patch_Count': net_zero_patch_count,
        'Net_Zero_Patch_pct': (100.0 * net_zero_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mixed_Sign_Patch_Count': mixed_sign_patch_count,
        'Mixed_Sign_Patch_pct': (100.0 * mixed_sign_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mixed_Sign_Net_Protective_Patch_Count': mixed_sign_net_protective_patch_count,
        'Mixed_Sign_Net_Protective_Patch_pct': (100.0 * mixed_sign_net_protective_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mixed_Sign_Net_Negative_Patch_Count': mixed_sign_net_negative_patch_count,
        'Mixed_Sign_Net_Negative_Patch_pct': (100.0 * mixed_sign_net_negative_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Positive_Only_Patch_Count': positive_only_patch_count,
        'Positive_Only_Patch_pct': (100.0 * positive_only_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Negative_Only_Patch_Count': negative_only_patch_count,
        'Negative_Only_Patch_pct': (100.0 * negative_only_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
    }
])

positive_ranking = mangrove_signed_profile.sort_values(
    ['Positive_Avoided_EAD_USD_attributed', 'Net_Avoided_EAD_USD_attributed'],
    ascending=[False, False],
).reset_index(drop=True)
negative_ranking = mangrove_signed_profile.loc[
    mangrove_signed_profile['Negative_Avoided_EAD_USD_attributed'] < 0
].sort_values('Negative_Avoided_EAD_USD_attributed', ascending=True).reset_index(drop=True)
absolute_ranking = mangrove_signed_profile.sort_values('Absolute_Avoided_EAD_USD_attributed', ascending=False).reset_index(drop=True)

mangrove_attribution_map = mangroves[mangrove_base_columns + ['geometry']].copy()
for mangrove_attribute_column in ['Parish', 'HECTARES', 'TYPE']:
    if mangrove_attribute_column in mangroves.columns and mangrove_attribute_column not in mangrove_attribution_map.columns:
        mangrove_attribution_map[mangrove_attribute_column] = mangroves[mangrove_attribute_column]
mangrove_attribution_map = mangrove_attribution_map.merge(mangrove_signed_profile, on='Mangrove_ID', how='left')
fill_zero_columns = [
    'Net_Avoided_EAD_USD_attributed',
    'Positive_Avoided_EAD_USD_attributed',
    'Negative_Avoided_EAD_USD_attributed',
    'Absolute_Avoided_EAD_USD_attributed',
    'Negative_Avoided_EAD_USD_attributed_abs',
    'Asset_Count',
    'Analysis_Unit_Count',
    'Positive_Asset_Count',
    'Negative_Asset_Count',
    'Buffer_Analysis_Unit_Count',
    'Fallback_Analysis_Unit_Count',
]
for summary_column in fill_zero_columns:
    mangrove_attribution_map[summary_column] = mangrove_attribution_map[summary_column].fillna(0.0)

run_summary = pandas.DataFrame([
    {
        'Scenario': SCENARIO,
        'Buffer_m': BUFFER_M,
        'Keep_Only_Nonzero_Avoided': KEEP_ONLY_NONZERO_AVOIDED,
        'Use_Nearest_Mangrove_Fallback_For_Outside_Buffer': USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER,
        'Edge_Segment_Length_m': EDGE_SEGMENT_LENGTH_M,
        'Input_Asset_Count': int(len(asset_status_all)),
        'Input_Analysis_Unit_Count': int(len(analysis_units_all)),
        'Input_Positive_Asset_Count': int((asset_status_all['Avoided_EAD_USD'] > 0).sum()),
        'Input_Negative_Asset_Count': int((asset_status_all['Avoided_EAD_USD'] < 0).sum()),
        'Buffer_Matched_Asset_Count': int(asset_status_all['Has_Buffer_Mangrove'].sum()),
        'Fallback_Asset_Count': int(asset_status_all['Used_Nearest_Fallback'].sum()),
        'Attributed_Asset_Count': int(asset_status_all['Has_Attributed_Mangrove'].sum()),
        'Input_Net_Avoided_EAD_USD': input_net_total_usd,
        'Input_Positive_Avoided_EAD_USD': input_positive_total_usd,
        'Input_Negative_Avoided_EAD_USD': input_negative_total_usd,
        'Attributed_Net_Avoided_EAD_USD': attributed_net_total_usd,
        'Attributed_Positive_Avoided_EAD_USD': attributed_positive_total_usd,
        'Attributed_Negative_Avoided_EAD_USD': attributed_negative_total_usd,
        'Unattributed_Net_Avoided_EAD_USD': unattributed_net_total_usd,
        'Fraction_Attributed_pct': (100.0 * attributed_net_total_usd / input_net_total_usd) if abs(input_net_total_usd) > 0 else numpy.nan,
        'Mangrove_Patch_Count_Total': total_mangrove_patch_count,
        'Mangrove_Patch_Count_Net_Protective': net_protective_patch_count,
        'Mangrove_Patch_pct_Net_Protective': (100.0 * net_protective_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mangrove_Patch_Count_Net_Negative': net_negative_patch_count,
        'Mangrove_Patch_pct_Net_Negative': (100.0 * net_negative_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mangrove_Patch_Count_Mixed_Sign': mixed_sign_patch_count,
        'Mangrove_Patch_pct_Mixed_Sign': (100.0 * mixed_sign_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mangrove_Patch_Count_Mixed_Sign_Net_Protective': mixed_sign_net_protective_patch_count,
        'Mangrove_Patch_pct_Mixed_Sign_Net_Protective': (100.0 * mixed_sign_net_protective_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Mangrove_Patch_Count_Mixed_Sign_Net_Negative': mixed_sign_net_negative_patch_count,
        'Mangrove_Patch_pct_Mixed_Sign_Net_Negative': (100.0 * mixed_sign_net_negative_patch_count / total_mangrove_patch_count) if total_mangrove_patch_count > 0 else numpy.nan,
        'Zero_Distance_Analysis_Unit_Count': int(matched_pairs_all.loc[matched_pairs_all['has_zero_distance_match'], 'Unit_ID'].nunique()) if len(matched_pairs_all) > 0 else 0,
        'Mean_Fallback_Distance_m': float(unit_to_mangrove_output.loc[unit_to_mangrove_output['Attribution_Source'] == 'nearest_outside_buffer', 'distance_m'].mean()) if (unit_to_mangrove_output['Attribution_Source'] == 'nearest_outside_buffer').any() else numpy.nan,
    }
])

unit_to_mangrove_csv = out_dir / f'analysis_unit_to_mangrove_attribution_{method_label}.csv'
asset_to_mangrove_csv = out_dir / f'asset_to_mangrove_attribution_{method_label}.csv'
asset_status_csv = out_dir / f'asset_attribution_status_{method_label}.csv'
layer_processing_csv = out_dir / f'layer_processing_summary_{method_label}.csv'
sector_attribution_breakdown_csv = out_dir / f'attribution_breakdown_by_sector_{method_label}.csv'
subsector_attribution_breakdown_csv = out_dir / f'attribution_breakdown_by_subsector_{method_label}.csv'
mangrove_total_csv = out_dir / f'mangrove_attribution_total_all_sectors_{method_label}.csv'
sector_signed_profile_csv = out_dir / f'sector_attribution_net_sign_profile_{method_label}.csv'
subsector_signed_profile_csv = out_dir / f'subsector_attribution_net_sign_profile_{method_label}.csv'
mangrove_signed_profile_csv = out_dir / f'mangrove_attribution_net_sign_profile_all_sectors_{method_label}.csv'
mangrove_patch_net_sign_summary_csv = out_dir / f'mangrove_patch_net_sign_summary_all_sectors_{method_label}.csv'
mangrove_sector_csv = out_dir / f'mangrove_attribution_by_sector_all_sectors_{method_label}.csv'
positive_ranking_csv = out_dir / f'mangrove_attribution_positive_ranking_all_sectors_{method_label}.csv'
negative_ranking_csv = out_dir / f'mangrove_attribution_negative_ranking_all_sectors_{method_label}.csv'
absolute_ranking_csv = out_dir / f'mangrove_attribution_absolute_ranking_all_sectors_{method_label}.csv'
run_summary_csv = out_dir / f'run_summary_all_sectors_{method_label}.csv'
mangrove_map_gpkg = out_dir / f'mangrove_attribution_total_all_sectors_{method_label}.gpkg'

unit_to_mangrove_output.to_csv(unit_to_mangrove_csv, index=False)
asset_to_mangrove_output.to_csv(asset_to_mangrove_csv, index=False)
asset_status_all.to_csv(asset_status_csv, index=False)
layer_processing_summary.to_csv(layer_processing_csv, index=False)
sector_attribution_breakdown.to_csv(sector_attribution_breakdown_csv, index=False)
subsector_attribution_breakdown.to_csv(subsector_attribution_breakdown_csv, index=False)
sector_signed_profile.to_csv(sector_signed_profile_csv, index=False)
subsector_signed_profile.to_csv(subsector_signed_profile_csv, index=False)
mangrove_signed_profile.to_csv(mangrove_total_csv, index=False)
mangrove_signed_profile.to_csv(mangrove_signed_profile_csv, index=False)
mangrove_patch_net_sign_summary.to_csv(mangrove_patch_net_sign_summary_csv, index=False)
mangrove_sector_summary.to_csv(mangrove_sector_csv, index=False)
positive_ranking.to_csv(positive_ranking_csv, index=False)
negative_ranking.to_csv(negative_ranking_csv, index=False)
absolute_ranking.to_csv(absolute_ranking_csv, index=False)
run_summary.to_csv(run_summary_csv, index=False)
mangrove_attribution_map.to_file(mangrove_map_gpkg, driver='GPKG')

print('Sector attribution breakdown (USD):')
display(sector_attribution_breakdown)
print('Subsector attribution breakdown (USD):')
display(subsector_attribution_breakdown)

print('Saved outputs:')
for output_path in [
    unit_to_mangrove_csv,
    asset_to_mangrove_csv,
    asset_status_csv,
    layer_processing_csv,
    sector_attribution_breakdown_csv,
    subsector_attribution_breakdown_csv,
    sector_signed_profile_csv,
    subsector_signed_profile_csv,
    mangrove_total_csv,
    mangrove_signed_profile_csv,
    mangrove_patch_net_sign_summary_csv,
    mangrove_sector_csv,
    positive_ranking_csv,
    negative_ranking_csv,
    absolute_ranking_csv,
    run_summary_csv,
    mangrove_map_gpkg,
]:
    print(f' - {output_path}')

In [ ]:
# Save one row per asset with patch count and mangrove IDs
if 'asset_to_mangrove_output' not in globals() or 'asset_status_all' not in globals():
    raise ValueError('Run the output-building cell first so asset_to_mangrove_output and asset_status_all exist.')

ordered_asset_pairs = asset_to_mangrove_output.copy()
ordered_asset_pairs['Mangrove_ID_Int'] = ordered_asset_pairs['Mangrove_ID'].astype(int)
ordered_asset_pairs = ordered_asset_pairs.sort_values(
    asset_key_columns + ['Absolute_Avoided_EAD_USD_attributed', 'Mangrove_ID_Int'],
    ascending=[True, True, True, True, True, False, True],
).reset_index(drop=True)

asset_absolute_attribution = (
    ordered_asset_pairs.groupby(asset_key_columns, as_index=False)['Absolute_Avoided_EAD_USD_attributed']
    .sum()
    .rename(columns={'Absolute_Avoided_EAD_USD_attributed': 'Absolute_Attributed_EAD_USD_Total'})
)
ordered_asset_pairs = ordered_asset_pairs.merge(asset_absolute_attribution, on=asset_key_columns, how='left')
ordered_asset_pairs['Absolute_Attributed_Share'] = numpy.where(
    ordered_asset_pairs['Absolute_Attributed_EAD_USD_Total'] > 0,
    ordered_asset_pairs['Absolute_Avoided_EAD_USD_attributed'] / ordered_asset_pairs['Absolute_Attributed_EAD_USD_Total'],
    numpy.nan,
)
ordered_asset_pairs['mangrove_share_label'] = ordered_asset_pairs.apply(
    lambda row: f"{int(row['Mangrove_ID_Int'])} ({row['Absolute_Attributed_Share']:.6f})" if pandas.notna(row['Absolute_Attributed_Share']) else f"{int(row['Mangrove_ID_Int'])} (nan)",
    axis=1,
)
ordered_asset_pairs['mangrove_attributed_usd_label'] = ordered_asset_pairs.apply(
    lambda row: f"{int(row['Mangrove_ID_Int'])} ({row['Avoided_EAD_USD_attributed']:.2f})",
    axis=1,
)

asset_patch_summary_core = (
    ordered_asset_pairs.groupby(asset_key_columns, as_index=False)
    .agg(
        Mangrove_Patch_Count=('Mangrove_ID_Int', 'nunique'),
        Mean_Distance_m=('Mean_Distance_m', 'mean'),
        Min_Distance_m=('Min_Distance_m', 'min'),
        Max_Distance_m=('Max_Distance_m', 'max'),
        Absolute_Attributed_EAD_USD_Total=('Absolute_Attributed_EAD_USD_Total', 'first'),
    )
)

mangrove_id_list = (
    ordered_asset_pairs.groupby(asset_key_columns)['Mangrove_ID_Int']
    .apply(lambda values: ', '.join(map(str, values.tolist())))
    .rename('Mangrove_ID_List')
    .reset_index()
)

mangrove_share_list = (
    ordered_asset_pairs.groupby(asset_key_columns)['mangrove_share_label']
    .apply(lambda values: '; '.join(values.tolist()))
    .rename('Mangrove_ID_Absolute_Share_List')
    .reset_index()
)

mangrove_attributed_usd_list = (
    ordered_asset_pairs.groupby(asset_key_columns)['mangrove_attributed_usd_label']
    .apply(lambda values: '; '.join(values.tolist()))
    .rename('Mangrove_ID_Attributed_USD_List')
    .reset_index()
)

attribution_source_list = (
    ordered_asset_pairs.groupby(asset_key_columns)
    .apply(
        lambda group: ', '.join([
            source_name
            for source_name, has_source in [
                ('buffer_area_distance', bool((group['Buffer_Analysis_Unit_Count'] > 0).any())),
                ('nearest_outside_buffer', bool((group['Fallback_Analysis_Unit_Count'] > 0).any())),
            ]
            if has_source
        ])
    )
    .rename('Attribution_Source_List')
    .reset_index()
)

asset_patch_summary = asset_status_all[[
    'Sector',
    'Subsector',
    'Asset',
    'Layer',
    'Asset_ID',
    'Avoided_EAD_USD',
    'Avoided_EAD_Sign',
    'Attributed_EAD_USD',
    'Attributed_EAD_Sign',
    'Unattributed_EAD_USD',
    'Has_Buffer_Mangrove',
    'Used_Nearest_Fallback',
    'Has_Attributed_Mangrove',
    'Analysis_Unit_Count',
]].copy()

for summary_table in [
    asset_patch_summary_core,
    mangrove_id_list,
    mangrove_share_list,
    mangrove_attributed_usd_list,
    attribution_source_list,
]:
    asset_patch_summary = asset_patch_summary.merge(summary_table, on=asset_key_columns, how='left')

asset_patch_summary['Mangrove_Patch_Count'] = asset_patch_summary['Mangrove_Patch_Count'].fillna(0).astype(int)
for string_column in [
    'Mangrove_ID_List',
    'Mangrove_ID_Absolute_Share_List',
    'Mangrove_ID_Attributed_USD_List',
    'Attribution_Source_List',
]:
    asset_patch_summary[string_column] = asset_patch_summary[string_column].fillna('')

asset_patch_summary['Absolute_Attributed_EAD_USD_Total'] = asset_patch_summary['Absolute_Attributed_EAD_USD_Total'].fillna(0.0)
asset_patch_summary['Absolute_Attributed_EAD_USD'] = asset_patch_summary['Attributed_EAD_USD'].abs()
asset_patch_summary = asset_patch_summary.sort_values(
    ['Mangrove_Patch_Count', 'Absolute_Attributed_EAD_USD', 'Asset_ID'],
    ascending=[False, False, True],
).reset_index(drop=True)

asset_patch_summary_csv = out_dir / f'asset_patch_count_and_ids_all_sectors_{method_label}.csv'
asset_patch_summary.to_csv(asset_patch_summary_csv, index=False)

print(f'Saved: {asset_patch_summary_csv}')
print('Patch-count summary across all sectors:')
display(asset_patch_summary['Mangrove_Patch_Count'].describe())
print('Top 15 assets by number of attributed mangrove patches:')
display(
    asset_patch_summary[[
        'Sector',
        'Subsector',
        'Asset_ID',
        'Attributed_EAD_USD',
        'Mangrove_Patch_Count',
        'Mangrove_ID_List',
        'Mangrove_ID_Attributed_USD_List',
    ]].head(15)
)


In [ ]:
# Review summary outputs
display(run_summary)
print('Top 20 mangrove patches by positive attributed avoided EAD (USD):')
display(positive_ranking.head(20))
print('Top 20 mangrove patches by negative attributed avoided EAD (USD, most negative first):')
display(negative_ranking.head(20))
print('Top 20 mangrove patches by absolute attributed avoided EAD (USD):')
display(absolute_ranking.head(20))

figure, axes = plt.subplots(ncols=3, figsize=(18, 7))

positive_plot_data = positive_ranking.head(12).sort_values('Positive_Avoided_EAD_USD_attributed')
axes[0].barh(
    positive_plot_data['Mangrove_ID'].astype(str),
    positive_plot_data['Positive_Avoided_EAD_USD_attributed'],
    color='#2f6f4f',
)
axes[0].set_title('Top positive patches')
axes[0].set_xlabel('Positive attributed avoided EAD (USD)')
axes[0].set_ylabel('Mangrove_ID')

negative_plot_data = negative_ranking.head(12).sort_values('Negative_Avoided_EAD_USD_attributed_abs')
axes[1].barh(
    negative_plot_data['Mangrove_ID'].astype(str),
    negative_plot_data['Negative_Avoided_EAD_USD_attributed_abs'],
    color='#b23a2f',
)
axes[1].set_title('Top negative patches')
axes[1].set_xlabel('Negative attributed avoided EAD magnitude (USD)')
axes[1].set_ylabel('Mangrove_ID')

absolute_plot_data = absolute_ranking.head(12).sort_values('Absolute_Avoided_EAD_USD_attributed')
axes[2].barh(
    absolute_plot_data['Mangrove_ID'].astype(str),
    absolute_plot_data['Absolute_Avoided_EAD_USD_attributed'],
    color='#4c6faf',
)
axes[2].set_title('Top absolute patches')
axes[2].set_xlabel('Absolute attributed avoided EAD (USD)')
axes[2].set_ylabel('Mangrove_ID')

plt.tight_layout()
plt.show()

## Top sectors and subsectors by attributed avoided EAD

In [ ]:
if 'sector_attribution_breakdown' not in globals() or 'subsector_attribution_breakdown' not in globals():
    raise ValueError('Run the output-building cell first so sector and subsector attribution tables exist.')


def format_compact_usd(value_usd: float) -> str:
    absolute_value_usd = abs(value_usd)
    if absolute_value_usd >= 1e6:
        return f"${value_usd / 1e6:,.2f}M"
    if absolute_value_usd >= 1e3:
        return f"${value_usd / 1e3:,.0f}k"
    return f"${value_usd:,.0f}"


sector_top_positive = (
    sector_attribution_breakdown.loc[sector_attribution_breakdown['Attributed_EAD_USD'] > 0]
    .sort_values('Attributed_EAD_USD', ascending=False)
    .head(6)
    .sort_values('Attributed_EAD_USD', ascending=True)
    .reset_index(drop=True)
)
subsector_top_positive = (
    subsector_attribution_breakdown.loc[subsector_attribution_breakdown['Attributed_EAD_USD'] > 0]
    .assign(Sector_Subsector=lambda data_frame: data_frame['Sector'] + ' — ' + data_frame['Subsector'])
    .sort_values('Attributed_EAD_USD', ascending=False)
    .head(8)
    .sort_values('Attributed_EAD_USD', ascending=True)
    .reset_index(drop=True)
)

figure, axes = plt.subplots(ncols=2, figsize=(16, 5.5))

axes[0].barh(
    sector_top_positive['Sector'],
    sector_top_positive['Attributed_EAD_USD'],
    color='#2f6f4f',
)
axes[0].set_title(f'Top sectors ({SCENARIO})')
axes[0].set_xlabel('Attributed avoided EAD (USD)')
axes[0].xaxis.set_major_formatter(FuncFormatter(lambda value, position: format_compact_usd(value)))

axes[1].barh(
    subsector_top_positive['Sector_Subsector'],
    subsector_top_positive['Attributed_EAD_USD'],
    color='#4c6faf',
)
axes[1].set_title(f'Top subsectors ({SCENARIO})')
axes[1].set_xlabel('Attributed avoided EAD (USD)')
axes[1].xaxis.set_major_formatter(FuncFormatter(lambda value, position: format_compact_usd(value)))

for axis, label_column in [
    (axes[0], 'Attributed_EAD_USD'),
    (axes[1], 'Attributed_EAD_USD'),
]:
    max_axis_value = float(axis.get_xlim()[1]) if axis.get_xlim()[1] > 0 else 0.0
    label_padding = max_axis_value * 0.01
    for container in axis.containers:
        for bar_value, bar_patch in zip(container.datavalues, container.patches):
            axis.text(
                bar_value + label_padding,
                bar_patch.get_y() + (bar_patch.get_height() / 2),
                format_compact_usd(bar_value),
                va='center',
                ha='left',
                fontsize=8,
            )

plt.tight_layout()
ranking_panel_png = out_dir / f'top_sector_subsector_attributed_avoided_ead_{method_label}.png'
figure.savefig(ranking_panel_png, dpi=300, bbox_inches='tight')
print(f'Saved: {ranking_panel_png}')
plt.show()

print('Top positive sectors by attributed avoided EAD (USD):')
display(sector_top_positive)
print('Top positive subsectors by attributed avoided EAD (USD):')
display(subsector_top_positive)


## Negative and net attribution profiles across sectors, subsectors, and mangrove patches

In [ ]:
if 'sector_signed_profile' not in globals() or 'subsector_signed_profile' not in globals() or 'mangrove_signed_profile' not in globals() or 'mangrove_patch_net_sign_summary' not in globals():
    raise ValueError('Run the output-building cell first so the signed profile tables exist.')
if 'format_compact_usd' not in globals():
    raise ValueError('Run the positive sector/subsector panel cell first so format_compact_usd exists.')

sector_negative_plot = sector_signed_profile.copy().sort_values(
    ['Negative_Attributed_EAD_USD_abs', 'Sector'],
    ascending=[True, True],
).reset_index(drop=True)
subsector_negative_plot = (
    subsector_signed_profile.assign(Sector_Subsector=lambda data_frame: data_frame['Sector'] + ' — ' + data_frame['Subsector'])
    .sort_values(['Negative_Attributed_EAD_USD_abs', 'Sector_Subsector'], ascending=[True, True])
    .reset_index(drop=True)
)

negative_figure_height = max(5.5, 0.42 * len(subsector_negative_plot))
negative_figure, negative_axes = plt.subplots(ncols=2, figsize=(16, negative_figure_height))

negative_axes[0].barh(
    sector_negative_plot['Sector'],
    sector_negative_plot['Negative_Attributed_EAD_USD_abs'],
    color='#b23a2f',
)
negative_axes[0].set_title(f'Negative attributed EAD by sector ({SCENARIO})')
negative_axes[0].set_xlabel('Negative attributed EAD magnitude (USD)')
negative_axes[0].xaxis.set_major_formatter(FuncFormatter(lambda value, position: format_compact_usd(value)))

negative_axes[1].barh(
    subsector_negative_plot['Sector_Subsector'],
    subsector_negative_plot['Negative_Attributed_EAD_USD_abs'],
    color='#cf6a61',
)
negative_axes[1].set_title(f'Negative attributed EAD by subsector ({SCENARIO})')
negative_axes[1].set_xlabel('Negative attributed EAD magnitude (USD)')
negative_axes[1].xaxis.set_major_formatter(FuncFormatter(lambda value, position: format_compact_usd(value)))

for axis in negative_axes:
    max_axis_value = float(axis.get_xlim()[1]) if axis.get_xlim()[1] > 0 else 0.0
    label_padding = max_axis_value * 0.01
    for container in axis.containers:
        for bar_value, bar_patch in zip(container.datavalues, container.patches):
            axis.text(
                bar_value + label_padding,
                bar_patch.get_y() + (bar_patch.get_height() / 2),
                f'-{format_compact_usd(bar_value)}',
                va='center',
                ha='left',
                fontsize=8,
            )

plt.tight_layout()
negative_panel_png = out_dir / f'negative_sector_subsector_attributed_ead_{method_label}.png'
negative_figure.savefig(negative_panel_png, dpi=300, bbox_inches='tight')
print(f'Saved: {negative_panel_png}')
plt.show()

sector_net_plot = sector_signed_profile.copy().sort_values(['Net_Attributed_EAD_USD', 'Sector'], ascending=[True, True]).reset_index(drop=True)
subsector_net_plot = (
    subsector_signed_profile.assign(Sector_Subsector=lambda data_frame: data_frame['Sector'] + ' — ' + data_frame['Subsector'])
    .sort_values(['Net_Attributed_EAD_USD', 'Sector_Subsector'], ascending=[True, True])
    .reset_index(drop=True)
)

net_figure_height = max(5.5, 0.42 * len(subsector_net_plot))
net_figure, net_axes = plt.subplots(ncols=2, figsize=(16, net_figure_height))

net_axes[0].barh(
    sector_net_plot['Sector'],
    sector_net_plot['Net_Attributed_EAD_USD'],
    color=numpy.where(sector_net_plot['Net_Attributed_EAD_USD'] >= 0, '#2f6f4f', '#b23a2f'),
)
net_axes[0].axvline(0, color='#6f6f6f', linewidth=0.8)
net_axes[0].set_title(f'Net attributed EAD by sector ({SCENARIO})')
net_axes[0].set_xlabel('Net attributed EAD (USD)')
net_axes[0].xaxis.set_major_formatter(FuncFormatter(lambda value, position: format_compact_usd(value)))

net_axes[1].barh(
    subsector_net_plot['Sector_Subsector'],
    subsector_net_plot['Net_Attributed_EAD_USD'],
    color=numpy.where(subsector_net_plot['Net_Attributed_EAD_USD'] >= 0, '#4c6faf', '#cf6a61'),
)
net_axes[1].axvline(0, color='#6f6f6f', linewidth=0.8)
net_axes[1].set_title(f'Net attributed EAD by subsector ({SCENARIO})')
net_axes[1].set_xlabel('Net attributed EAD (USD)')
net_axes[1].xaxis.set_major_formatter(FuncFormatter(lambda value, position: format_compact_usd(value)))

for axis in net_axes:
    x_limits = axis.get_xlim()
    label_padding = (x_limits[1] - x_limits[0]) * 0.01
    for container in axis.containers:
        for bar_value, bar_patch in zip(container.datavalues, container.patches):
            if bar_value >= 0:
                text_x = bar_value + label_padding
                text_alignment = 'left'
            else:
                text_x = bar_value - label_padding
                text_alignment = 'right'
            axis.text(
                text_x,
                bar_patch.get_y() + (bar_patch.get_height() / 2),
                format_compact_usd(bar_value),
                va='center',
                ha=text_alignment,
                fontsize=8,
            )

plt.tight_layout()
net_panel_png = out_dir / f'net_sector_subsector_attributed_ead_{method_label}.png'
net_figure.savefig(net_panel_png, dpi=300, bbox_inches='tight')
print(f'Saved: {net_panel_png}')
plt.show()

sector_net_overview = sector_signed_profile[[
    'Sector',
    'Positive_Avoided_EAD_USD',
    'Negative_Avoided_EAD_USD',
    'Net_Avoided_EAD_USD',
    'Positive_share_of_gross_avoided_pct',
    'Positive_Attributed_EAD_USD',
    'Negative_Attributed_EAD_USD',
    'Net_Attributed_EAD_USD',
    'Positive_share_of_gross_attributed_pct',
    'Negative_share_of_gross_attributed_pct',
    'Net_retention_of_positive_attributed_pct',
    'Negative_drag_on_positive_attributed_pct',
    'Has_Mixed_Positive_And_Negative_Assets',
    'Net_Impact_Class',
]].copy()
subsector_net_overview = subsector_signed_profile[[
    'Sector',
    'Subsector',
    'Positive_Avoided_EAD_USD',
    'Negative_Avoided_EAD_USD',
    'Net_Avoided_EAD_USD',
    'Positive_share_of_gross_avoided_pct',
    'Positive_Attributed_EAD_USD',
    'Negative_Attributed_EAD_USD',
    'Net_Attributed_EAD_USD',
    'Positive_share_of_gross_attributed_pct',
    'Negative_share_of_gross_attributed_pct',
    'Net_retention_of_positive_attributed_pct',
    'Negative_drag_on_positive_attributed_pct',
    'Has_Mixed_Positive_And_Negative_Assets',
    'Net_Impact_Class',
]].copy()
mangrove_net_overview = mangrove_signed_profile[[
    'Mangrove_ID',
    'Positive_Avoided_EAD_USD_attributed',
    'Negative_Avoided_EAD_USD_attributed',
    'Net_Avoided_EAD_USD_attributed',
    'Positive_share_of_gross_attributed_pct',
    'Negative_share_of_gross_attributed_pct',
    'Net_retention_of_positive_attributed_pct',
    'Negative_drag_on_positive_attributed_pct',
    'Absolute_Avoided_EAD_USD_attributed',
    'Positive_Attributed_Asset_Count',
    'Negative_Attributed_Asset_Count',
    'Has_Mixed_Positive_And_Negative_Assets',
    'Net_Impact_Class',
]].copy().sort_values(
    ['Absolute_Avoided_EAD_USD_attributed', 'Mangrove_ID'],
    ascending=[False, True],
).reset_index(drop=True)

print('Sector net sign profile (USD):')
display(sector_net_overview)
print('Subsector net sign profile (USD):')
display(subsector_net_overview)
print('Mangrove patch sign summary by patch count:')
display(mangrove_patch_net_sign_summary)
print('Mangrove patch net sign profile (top 25 by absolute attributed EAD):')
display(mangrove_net_overview.head(25))
print('Mangrove patches with both positive and negative attributed assets:')
display(mangrove_net_overview.loc[mangrove_net_overview['Has_Mixed_Positive_And_Negative_Assets']].head(25))


In [ ]:
# Map mangrove patches by net attributed avoided EAD (USD) across all sectors
if 'mangrove_attribution_map' not in globals():
    raise ValueError('Run the summary/output cell first so mangrove_attribution_map exists.')

mangrove_plot = mangrove_attribution_map.copy()
if mangrove_plot.crs is None:
    raise ValueError('Mangrove attribution map CRS is missing.')
if str(mangrove_plot.crs).upper() != 'EPSG:3448':
    mangrove_plot = mangrove_plot.to_crs('EPSG:3448')

value_column = 'Net_Avoided_EAD_USD_attributed'
if value_column not in mangrove_plot.columns:
    raise KeyError(f"Column '{value_column}' not found in mangrove attribution map.")

values = mangrove_plot[value_column].fillna(0.0)
absolute_values = values.abs()
display_cap = float(absolute_values.quantile(MAP_DISPLAY_QUANTILE))
if display_cap <= 0:
    display_cap = float(absolute_values.max()) if float(absolute_values.max()) > 0 else 1.0

mangrove_plot['_plot_value'] = values.clip(lower=-display_cap, upper=display_cap)

red_white_green_colormap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)
value_norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

figure, axis = plt.subplots(figsize=(10.8, 9.4))
axis.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=axis, color='#9a9a9a', linewidth=0.5, zorder=1)

mangrove_plot.plot(
    ax=axis,
    column='_plot_value',
    cmap=red_white_green_colormap,
    norm=value_norm,
    edgecolor='#6f6f6f',
    linewidth=0.35,
    alpha=0.98,
    zorder=2,
)

scalar_mappable = ScalarMappable(norm=value_norm, cmap=red_white_green_colormap)
scalar_mappable.set_array([])
colorbar = figure.colorbar(
    scalar_mappable,
    ax=axis,
    orientation='horizontal',
    fraction=0.045,
    pad=0.02,
)
colorbar.set_label('Net attributed avoided EAD (USD) | red = negative, green = positive')
colorbar.ax.xaxis.set_major_formatter(FuncFormatter(lambda tick_value, tick_position: f'{tick_value:,.0f}'))

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

fallback_title_suffix = ' + nearest fallback' if USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER else ''
axis.set_title(
    f'All sectors: net mangrove attributed avoided EADs ({SCENARIO} scenario, {int(round(BUFFER_M))} m buffer{fallback_title_suffix})',
    fontsize=12,
)
axis.set_axis_off()
plt.tight_layout()

map_png = out_dir / f'mangrove_attribution_map_all_sectors_{method_label}_q{int(MAP_DISPLAY_QUANTILE * 1000)}.png'
figure.savefig(map_png, dpi=300, bbox_inches='tight')
print(f'Saved: {map_png}')
plt.show()